# 1. MRI DICOM extraction and inventory verification

This notebook starts from the clean 90-day MRI extraction manifest created in the previous notebook.

The cohort is already fixed at 1063 selected baseline or near-baseline T1 MRI images. This notebook should not rebuild labels or reselect MRI scans. Its purpose is to extract only the selected DICOM series from the downloaded zip files and prepare them for DICOM-to-NIfTI conversion.

## 1.1. Why I Need to Convert DICOM to NIfTI

The MRI images downloaded from ADNI are stored in DICOM format. DICOM is the standard medical imaging format produced by scanners and used in clinical systems. In my downloaded ADNI files, each selected MRI image ID usually corresponds to a DICOM series, meaning a folder containing many `.dcm` files.

For example, one MRI scan may be stored as 160 separate DICOM files, where each file represents one slice of the 3D MRI volume. Some scans may be stored differently, including as a single large DICOM file.

For neuroimaging analysis and machine learning, it is more practical to work with NIfTI files. NIfTI stores the MRI as a single 3D brain volume, usually with the extension `.nii` or `.nii.gz`.

Therefore, this notebook will first extract the selected DICOM series from the ADNI zip files and then convert each selected series into NIfTI format. The resulting NIfTI files will be used in the later MRI preprocessing and modelling steps.

In [ ]:
from google.colab import drive
from pathlib import Path
import pandas as pd
import zipfile
import re
import os
import numpy as np
import time
import subprocess

In [ ]:
drive.mount("/content/drive", force_remount=True)

BASE_DIR = Path("/content/drive/My Drive/adni_mri")

MANIFEST_DIR = BASE_DIR / "manifest"
RAW_DOWNLOADS_DIR = BASE_DIR / "raw_downloads"
DICOM_DIR = BASE_DIR / "dicom_selected_90d"
NIFTI_DIR = BASE_DIR / "nifti"
QC_DIR = BASE_DIR / "qc"

EXTRACTION_MANIFEST_PATH = MANIFEST_DIR / "clean_90d_extraction_manifest_1063.csv"

print("Base folder exists:", BASE_DIR.exists())
print("Manifest folder exists:", MANIFEST_DIR.exists())
print("Raw downloads folder exists:", RAW_DOWNLOADS_DIR.exists())
print("DICOM output folder exists:", DICOM_DIR.exists())
print("NIfTI folder exists:", NIFTI_DIR.exists())
print("QC folder exists:", QC_DIR.exists())

print("\nExtraction manifest exists:", EXTRACTION_MANIFEST_PATH.exists())

print("\nRaw download zip files:")
for file in sorted(RAW_DOWNLOADS_DIR.glob("*.zip")):
    size_gb = file.stat().st_size / (1024 ** 3)
    print(f"- {file.name} | {size_gb:.2f} GB")

## 1.2. Load the Extraction Manifest

This notebook starts from the cleaned extraction manifest created in the previous notebook. use this file as the source of truth for preprocessing.

Each row represents one selected MRI scan from the 90-day baseline cohort. The manifest tells me which subject the scan belongs to, what the final clinical label is, which ADNI image ID should be extracted, and which zip file contains the DICOM files.

In [ ]:
extraction_manifest = pd.read_csv(EXTRACTION_MANIFEST_PATH)

print("Extraction manifest shape:", extraction_manifest.shape)

print("\nColumns:")
print(extraction_manifest.columns.tolist())

print("\nFinal group counts:")
print(extraction_manifest["final_group"].value_counts())

print("\nSource zip counts:")
print(extraction_manifest["source_zip"].value_counts())

print("\nBasic checks:")
print("Rows:", len(extraction_manifest))
print("Unique RIDs:", extraction_manifest["RID"].nunique())
print("Unique subjects:", extraction_manifest["subject_id"].nunique())
print("Unique image IDs:", extraction_manifest["image_id"].nunique())
print("Missing image IDs:", extraction_manifest["image_id"].isna().sum())
print("Missing source zips:", extraction_manifest["source_zip"].isna().sum())

print("\nPreview:")
display(extraction_manifest.head())

## 1.3. Create DICOM Extraction Plan

The extraction manifest contains one selected MRI image ID per subject. Before extracting files from the downloaded zip archives, The notebook creates an extraction plan that maps each selected image ID to its source zip file and intended output folder.

The planned folder structure keeps the extracted DICOM files organised by final clinical group, subject ID, and ADNI image ID. This helps keep the preprocessing workflow traceable and makes it easier to check which files belong to each subject.

In [ ]:
DICOM_EXTRACTION_PLAN_PATH = QC_DIR / "clean_90d_dicom_extraction_plan_1063.csv"

DICOM_DIR.mkdir(parents=True, exist_ok=True)
NIFTI_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)

extraction_plan = extraction_manifest.copy()

extraction_plan["image_id"] = extraction_plan["image_id"].astype(int)

extraction_plan["zip_path"] = extraction_plan["source_zip"].apply(
    lambda x: str(RAW_DOWNLOADS_DIR / x)
)

extraction_plan["zip_exists"] = extraction_plan["zip_path"].apply(
    lambda x: Path(x).exists()
)

extraction_plan["dicom_output_dir"] = extraction_plan.apply(
    lambda row: str(
        DICOM_DIR
        / row["final_group"]
        / row["subject_id"]
        / f"I{int(row['image_id'])}"
    ),
    axis=1
)

extraction_plan["output_dir_already_exists"] = extraction_plan["dicom_output_dir"].apply(
    lambda x: Path(x).exists()
)

for group in sorted(extraction_plan["final_group"].unique()):
    (DICOM_DIR / group).mkdir(parents=True, exist_ok=True)

extraction_plan.to_csv(DICOM_EXTRACTION_PLAN_PATH, index=False)

print("Saved DICOM extraction plan:")
print(DICOM_EXTRACTION_PLAN_PATH)

print("\nExtraction plan shape:")
print(extraction_plan.shape)

print("\nZip file availability:")
print(extraction_plan[["source_zip", "zip_exists"]].drop_duplicates().sort_values("source_zip"))

print("\nImages planned for extraction by source zip:")
print(extraction_plan["source_zip"].value_counts())

print("\nImages planned for extraction by final group:")
print(extraction_plan["final_group"].value_counts())

print("\nOutput folders that already exist:")
print(extraction_plan["output_dir_already_exists"].value_counts())

assert extraction_plan["zip_exists"].all(), "At least one required zip file is missing."

print("\nPreview:")
display(
    extraction_plan[
        [
            "RID",
            "subject_id",
            "final_group",
            "image_id",
            "source_zip",
            "total_dcm_file_count",
            "dicom_output_dir"
        ]
    ].head()
)

## 1.4. Pilot DICOM Extraction

Before extracting the full cohort, run a small pilot extraction. This step extracts a few selected MRI image IDs from the downloaded zip files and saves them into the planned DICOM folder structure.

The purpose of this pilot is to verify that the code correctly finds the selected image IDs inside the zip archives, extracts the expected number of DICOM files, and stores them in the correct subject-level folders.

In [ ]:
import shutil

PILOT_EXTRACTION_LOG_PATH = QC_DIR / "pilot_dicom_extraction_log.csv"

# Select a small pilot set:
# one image from each source zip, plus one single-file DICOM case if available
pilot_parts = [
    extraction_plan.groupby("source_zip", group_keys=False).head(1),
    extraction_plan[extraction_plan["total_dcm_file_count"] == 1].head(1)
]

pilot_plan = (
    pd.concat(pilot_parts, axis=0)
    .drop_duplicates(subset=["image_id"])
    .copy()
)

pilot_plan["image_id"] = pilot_plan["image_id"].astype(int)

print("Pilot images selected:", len(pilot_plan))
display(
    pilot_plan[
        [
            "RID",
            "subject_id",
            "final_group",
            "image_id",
            "source_zip",
            "total_dcm_file_count",
            "description",
            "dicom_output_dir"
        ]
    ]
)

pilot_lookup = pilot_plan.set_index("image_id").to_dict(orient="index")
pilot_ids = set(pilot_plan["image_id"].astype(int))

pilot_extraction_records = []

for zip_name in sorted(pilot_plan["source_zip"].unique()):
    zip_path = RAW_DOWNLOADS_DIR / zip_name
    ids_in_zip = set(
        pilot_plan.loc[pilot_plan["source_zip"] == zip_name, "image_id"].astype(int)
    )

    print(f"\nProcessing {zip_name}...")

    with zipfile.ZipFile(zip_path, "r") as z:
        extracted_counts = {image_id: 0 for image_id in ids_in_zip}

        for info in z.infolist():
            internal_path = info.filename

            if info.is_dir():
                continue

            if not internal_path.lower().endswith(".dcm"):
                continue

            matches = re.findall(r"I(\d+)", internal_path)

            if len(matches) == 0:
                continue

            image_id = int(matches[-1])

            if image_id not in ids_in_zip:
                continue

            row = pilot_lookup[image_id]
            output_dir = Path(row["dicom_output_dir"])
            output_dir.mkdir(parents=True, exist_ok=True)

            extracted_counts[image_id] += 1
            original_name = Path(internal_path).name
            output_name = f"{extracted_counts[image_id]:04d}_{original_name}"
            output_path = output_dir / output_name

            with z.open(info) as source_file:
                with open(output_path, "wb") as target_file:
                    shutil.copyfileobj(source_file, target_file)

            pilot_extraction_records.append({
                "image_id": image_id,
                "source_zip": zip_name,
                "internal_path": internal_path,
                "output_path": str(output_path),
                "file_size_mb": info.file_size / (1024 ** 2)
            })

    print("Extracted counts from this zip:")
    print(extracted_counts)

pilot_extraction_log = pd.DataFrame(pilot_extraction_records)
pilot_extraction_log.to_csv(PILOT_EXTRACTION_LOG_PATH, index=False)

pilot_summary = (
    pilot_extraction_log
    .groupby(["image_id", "source_zip"])
    .size()
    .reset_index(name="extracted_dcm_file_count")
    .merge(
        pilot_plan[
            [
                "image_id",
                "subject_id",
                "final_group",
                "total_dcm_file_count",
                "dicom_output_dir"
            ]
        ],
        on="image_id",
        how="left"
    )
)

pilot_summary["matches_expected_count"] = (
    pilot_summary["extracted_dcm_file_count"]
    == pilot_summary["total_dcm_file_count"]
)

print("\nSaved pilot extraction log:")
print(PILOT_EXTRACTION_LOG_PATH)

print("\nPilot extraction summary:")
display(pilot_summary)

print("\nAll pilot counts match expected counts:")
print(pilot_summary["matches_expected_count"].all())

## 1.5. Extract the Full 90-Day DICOM Cohort

The pilot extraction confirmed that the selected image IDs can be correctly found inside the ADNI zip archives and extracted into the planned folder structure.

In this step, I extract all 1063 selected DICOM series. The extraction is organised by final clinical group, subject ID, and ADNI image ID. The code is written to be resumable: if an output folder already contains the expected number of DICOM files, it will be skipped rather than extracted again.

In [ ]:
FULL_EXTRACTION_REPORT_PATH = QC_DIR / "clean_90d_full_dicom_extraction_report.csv"

def count_existing_dicoms(folder_path):
    folder_path = Path(folder_path)
    if not folder_path.exists():
        return 0
    return len(list(folder_path.rglob("*.dcm")))

full_plan = extraction_plan.copy()
full_plan["image_id"] = full_plan["image_id"].astype(int)
full_plan["expected_dcm_file_count"] = full_plan["total_dcm_file_count"].astype(int)

full_plan["existing_dcm_file_count_before"] = full_plan["dicom_output_dir"].apply(
    count_existing_dicoms
)

already_complete_ids = set(
    full_plan.loc[
        full_plan["existing_dcm_file_count_before"] == full_plan["expected_dcm_file_count"],
        "image_id"
    ].astype(int)
)

to_extract_plan = full_plan[
    ~full_plan["image_id"].isin(already_complete_ids)
].copy()

print("Total selected MRI series:", len(full_plan))
print("Already complete before extraction:", len(already_complete_ids))
print("Remaining to extract:", len(to_extract_plan))

# Remove incomplete output folders only for images that need re-extraction
for output_dir in to_extract_plan["dicom_output_dir"]:
    output_dir = Path(output_dir)
    if output_dir.exists():
        shutil.rmtree(output_dir)

extract_lookup = to_extract_plan.set_index("image_id").to_dict(orient="index")
ids_to_extract = set(to_extract_plan["image_id"].astype(int))

start_time = time.time()

extracted_counts = {image_id: 0 for image_id in ids_to_extract}

for zip_name in sorted(to_extract_plan["source_zip"].unique()):
    zip_path = RAW_DOWNLOADS_DIR / zip_name

    ids_in_zip = set(
        to_extract_plan.loc[
            to_extract_plan["source_zip"] == zip_name,
            "image_id"
        ].astype(int)
    )

    print(f"\nProcessing {zip_name}")
    print("Images to extract from this zip:", len(ids_in_zip))

    with zipfile.ZipFile(zip_path, "r") as z:
        for info in z.infolist():
            internal_path = info.filename

            if info.is_dir():
                continue

            if not internal_path.lower().endswith(".dcm"):
                continue

            matches = re.findall(r"I(\d+)", internal_path)

            if len(matches) == 0:
                continue

            image_id = int(matches[-1])

            if image_id not in ids_in_zip:
                continue

            row = extract_lookup[image_id]
            output_dir = Path(row["dicom_output_dir"])
            output_dir.mkdir(parents=True, exist_ok=True)

            extracted_counts[image_id] += 1

            original_name = Path(internal_path).name
            output_name = f"{extracted_counts[image_id]:04d}_{original_name}"
            output_path = output_dir / output_name

            with z.open(info) as source_file:
                with open(output_path, "wb") as target_file:
                    shutil.copyfileobj(source_file, target_file)

    extracted_from_zip = sum(
        extracted_counts[image_id] for image_id in ids_in_zip
    )

    print("DICOM files extracted from this zip:", extracted_from_zip)

full_plan["existing_dcm_file_count_after"] = full_plan["dicom_output_dir"].apply(
    count_existing_dicoms
)

full_plan["extraction_status"] = np.where(
    full_plan["existing_dcm_file_count_after"] == full_plan["expected_dcm_file_count"],
    "complete",
    np.where(
        full_plan["existing_dcm_file_count_after"] == 0,
        "not_extracted",
        "count_mismatch"
    )
)

full_plan["was_already_complete_before"] = full_plan["image_id"].isin(already_complete_ids)

full_plan.to_csv(FULL_EXTRACTION_REPORT_PATH, index=False)

elapsed_minutes = (time.time() - start_time) / 60

print("\nSaved full extraction report:")
print(FULL_EXTRACTION_REPORT_PATH)

print("\nExtraction status counts:")
print(full_plan["extraction_status"].value_counts())

print("\nAlready complete before extraction:")
print(full_plan["was_already_complete_before"].value_counts())

print("\nElapsed time in minutes:")
print(round(elapsed_minutes, 2))

print("\nCount mismatches, if any:")
count_mismatches = full_plan[full_plan["extraction_status"] != "complete"].copy()
print(len(count_mismatches))

if len(count_mismatches) > 0:
    display(
        count_mismatches[
            [
                "RID",
                "subject_id",
                "final_group",
                "image_id",
                "source_zip",
                "expected_dcm_file_count",
                "existing_dcm_file_count_after",
                "dicom_output_dir",
                "extraction_status"
            ]
        ]
    )

print("\nAll selected DICOM series extracted successfully:")
print((full_plan["extraction_status"] == "complete").all())

## 1.6. Prepare DICOM-to-NIfTI Conversion

All 1063 selected DICOM series were successfully extracted from the ADNI zip files. The next step is to convert each extracted DICOM series into NIfTI format.

The DICOM folders are now stored by final clinical group, subject ID, and ADNI image ID. The NIfTI outputs will be saved separately so that they can be used in later MRI preprocessing and modelling steps.

In [ ]:
# dcm2niix is not a Python library.
# It is an external command-line tool used to convert DICOM MRI series
# into NIfTI files, usually saved as .nii or .nii.gz.
#
# In this notebook, the workflow will be:
# extracted DICOM folder -> dcm2niix -> NIfTI MRI volume

# Check whether dcm2niix is already installed in the current Colab runtime.
# The "which" command searches for the executable program.
result = subprocess.run(
    ["which", "dcm2niix"],
    capture_output=True,
    text=True
)

# If dcm2niix is not found, install it using apt-get.
# This installs the tool only in the current Colab session.
# If the runtime restarts, it may need to be installed again.
if result.returncode != 0:
    print("dcm2niix is not installed. Installing now...")
    !apt-get update -qq
    !apt-get install -y dcm2niix
else:
    print("dcm2niix is already installed:")
    print(result.stdout.strip())

# Print the first few lines of the dcm2niix help text.
# This confirms that the command is available and working.
print("\ndcm2niix version check:")
!dcm2niix -h | head -n 5

# Make sure the main NIfTI output folder exists.
# Converted MRI files will be saved here later.
NIFTI_DIR.mkdir(parents=True, exist_ok=True)

# Create one NIfTI output subfolder per final clinical group.
# This keeps the converted files organised by label.
for group in sorted(extraction_manifest["final_group"].unique()):
    (NIFTI_DIR / group).mkdir(parents=True, exist_ok=True)

print("\nNIfTI output folder:")
print(NIFTI_DIR)

print("\nNIfTI group folders:")
for folder in sorted(NIFTI_DIR.iterdir()):
    if folder.is_dir():
        print("-", folder)

## 1.7. Pilot DICOM-to-NIfTI Conversion

Before converting the full cohort, test DICOM-to-NIfTI conversion on a small pilot set. The pilot includes examples from the extracted DICOM folders and also includes one single-file DICOM series if available.

This step checks whether `dcm2niix` can successfully read the extracted DICOM folders and produce compressed NIfTI files (`.nii.gz`) and sidecar metadata files (`.json`). If the pilot conversion works, the same logic can be applied to the full 1063-image cohort.

In [ ]:
PILOT_CONVERSION_REPORT_PATH = QC_DIR / "pilot_dicom_to_nifti_conversion_report.csv"

# Select a small pilot set:
# - one image from each source zip
# - plus one single-file DICOM case, if available
pilot_conversion_parts = [
    extraction_plan.groupby("source_zip", group_keys=False).head(1),
    extraction_plan[extraction_plan["total_dcm_file_count"] == 1].head(1)
]

pilot_conversion_plan = (
    pd.concat(pilot_conversion_parts, axis=0)
    .drop_duplicates(subset=["image_id"])
    .copy()
)

pilot_conversion_plan["image_id"] = pilot_conversion_plan["image_id"].astype(int)

print("Pilot conversion images selected:", len(pilot_conversion_plan))

display(
    pilot_conversion_plan[
        [
            "RID",
            "subject_id",
            "final_group",
            "image_id",
            "source_zip",
            "total_dcm_file_count",
            "description",
            "dicom_output_dir"
        ]
    ]
)

pilot_conversion_records = []

for _, row in pilot_conversion_plan.iterrows():
    subject_id = row["subject_id"]
    final_group = row["final_group"]
    image_id = int(row["image_id"])

    # This is the folder containing the extracted DICOM files for this image ID.
    input_dicom_dir = Path(row["dicom_output_dir"])

    # Each NIfTI output is saved into the folder matching the final clinical label.
    output_nifti_dir = NIFTI_DIR / final_group
    output_nifti_dir.mkdir(parents=True, exist_ok=True)

    # This filename keeps the subject ID and ADNI image ID traceable.
    output_file_base = f"{subject_id}_I{image_id}"

    # dcm2niix command:
    # -z y  : compress output as .nii.gz
    # -o    : output folder
    # -f    : output filename pattern
    # input : extracted DICOM folder
    command = [
        "dcm2niix",
        "-z", "y",
        "-o", str(output_nifti_dir),
        "-f", output_file_base,
        str(input_dicom_dir)
    ]

    print(f"\nConverting {subject_id}, image I{image_id}...")

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    expected_nifti_path = output_nifti_dir / f"{output_file_base}.nii.gz"
    expected_json_path = output_nifti_dir / f"{output_file_base}.json"

    pilot_conversion_records.append({
        "RID": row["RID"],
        "subject_id": subject_id,
        "final_group": final_group,
        "image_id": image_id,
        "input_dicom_dir": str(input_dicom_dir),
        "output_nifti_dir": str(output_nifti_dir),
        "output_file_base": output_file_base,
        "return_code": result.returncode,
        "nifti_exists": expected_nifti_path.exists(),
        "json_exists": expected_json_path.exists(),
        "nifti_path": str(expected_nifti_path),
        "json_path": str(expected_json_path),
        "stdout_tail": result.stdout[-1000:],
        "stderr_tail": result.stderr[-1000:]
    })

pilot_conversion_report = pd.DataFrame(pilot_conversion_records)
pilot_conversion_report.to_csv(PILOT_CONVERSION_REPORT_PATH, index=False)

print("\nSaved pilot conversion report:")
print(PILOT_CONVERSION_REPORT_PATH)

print("\nPilot conversion status:")
display(
    pilot_conversion_report[
        [
            "subject_id",
            "final_group",
            "image_id",
            "return_code",
            "nifti_exists",
            "json_exists",
            "nifti_path"
        ]
    ]
)

print("\nAll pilot NIfTI files created:")
print(pilot_conversion_report["nifti_exists"].all())

print("\nAll pilot JSON files created:")
print(pilot_conversion_report["json_exists"].all())

### 1.7.1. Interpretation of Pilot DICOM-to-NIfTI Conversion

The pilot DICOM-to-NIfTI conversion was successful. Four selected MRI series were converted using `dcm2niix`, including examples from each source zip file and one single-file DICOM case.

All four conversions returned code `0`, which indicates that `dcm2niix` completed without errors. For each pilot case, both the compressed NIfTI file (`.nii.gz`) and the accompanying JSON metadata file were created successfully.

This confirms that the extracted DICOM folders can be read by `dcm2niix` and converted into NIfTI format. Importantly, the single-file DICOM case also converted successfully, supporting the earlier interpretation that these one-file DICOM series are valid large DICOM volumes rather than incomplete downloads.

Based on this result, the same conversion logic can now be applied to the full clean 90-day MRI cohort of 1063 selected images.

## 1.8. Convert the Full 90-Day DICOM Cohort to NIfTI

The pilot conversion confirmed that `dcm2niix` can successfully convert the extracted DICOM folders into compressed NIfTI files and JSON metadata files.

In this step, I apply the same conversion process to all 1063 selected MRI series in the clean 90-day cohort. The conversion is resumable: if a NIfTI file already exists for a subject and image ID, that case is skipped. This is useful because the pilot conversions have already created a few files, and it also protects the workflow if the runtime is interrupted.

The converted files are saved into group-level folders under the main NIfTI directory.

In [ ]:
FULL_CONVERSION_REPORT_PATH = QC_DIR / "clean_90d_full_dicom_to_nifti_conversion_report.csv"

def find_existing_nifti_files(output_dir, output_file_base):
    """
    dcm2niix usually creates:
        output_file_base.nii.gz
        output_file_base.json

    However, in rare cases it may add suffixes if it detects multiple outputs.
    This function checks for any NIfTI files matching the intended base name.
    """
    output_dir = Path(output_dir)
    return sorted(output_dir.glob(f"{output_file_base}*.nii*"))

def find_existing_json_files(output_dir, output_file_base):
    """
    Check for JSON sidecar metadata files created by dcm2niix.
    """
    output_dir = Path(output_dir)
    return sorted(output_dir.glob(f"{output_file_base}*.json"))

conversion_plan = extraction_plan.copy()
conversion_plan["image_id"] = conversion_plan["image_id"].astype(int)

conversion_records = []

start_time = time.time()

print("Total MRI series planned for conversion:", len(conversion_plan))

for index, row in conversion_plan.iterrows():
    subject_id = row["subject_id"]
    final_group = row["final_group"]
    image_id = int(row["image_id"])

    input_dicom_dir = Path(row["dicom_output_dir"])
    output_nifti_dir = NIFTI_DIR / final_group
    output_nifti_dir.mkdir(parents=True, exist_ok=True)

    output_file_base = f"{subject_id}_I{image_id}"

    existing_nifti_files_before = find_existing_nifti_files(
        output_nifti_dir,
        output_file_base
    )

    existing_json_files_before = find_existing_json_files(
        output_nifti_dir,
        output_file_base
    )

    # If the NIfTI file already exists, skip conversion for this image.
    # This makes the process resumable and avoids reconverting the pilot files.
    if len(existing_nifti_files_before) > 0:
        conversion_records.append({
            "RID": row["RID"],
            "subject_id": subject_id,
            "final_group": final_group,
            "image_id": image_id,
            "input_dicom_dir": str(input_dicom_dir),
            "output_nifti_dir": str(output_nifti_dir),
            "output_file_base": output_file_base,
            "conversion_status": "skipped_existing_nifti",
            "return_code": None,
            "nifti_exists": True,
            "json_exists": len(existing_json_files_before) > 0,
            "n_nifti_files": len(existing_nifti_files_before),
            "n_json_files": len(existing_json_files_before),
            "nifti_files": "; ".join(str(path) for path in existing_nifti_files_before),
            "json_files": "; ".join(str(path) for path in existing_json_files_before),
            "stdout_tail": "",
            "stderr_tail": ""
        })
        continue

    # Check that the extracted DICOM input folder exists before conversion.
    if not input_dicom_dir.exists():
        conversion_records.append({
            "RID": row["RID"],
            "subject_id": subject_id,
            "final_group": final_group,
            "image_id": image_id,
            "input_dicom_dir": str(input_dicom_dir),
            "output_nifti_dir": str(output_nifti_dir),
            "output_file_base": output_file_base,
            "conversion_status": "missing_dicom_input_dir",
            "return_code": None,
            "nifti_exists": False,
            "json_exists": False,
            "n_nifti_files": 0,
            "n_json_files": 0,
            "nifti_files": "",
            "json_files": "",
            "stdout_tail": "",
            "stderr_tail": ""
        })
        continue

    command = [
        "dcm2niix",
        "-z", "y",                       # save compressed .nii.gz output
        "-o", str(output_nifti_dir),      # output folder
        "-f", output_file_base,           # output filename base
        str(input_dicom_dir)              # input DICOM folder
    ]

    print(f"[{index + 1}/{len(conversion_plan)}] Converting {subject_id}, image I{image_id}")

    result = subprocess.run(
        command,
        capture_output=True,
        text=True
    )

    nifti_files_after = find_existing_nifti_files(
        output_nifti_dir,
        output_file_base
    )

    json_files_after = find_existing_json_files(
        output_nifti_dir,
        output_file_base
    )

    nifti_exists = len(nifti_files_after) > 0
    json_exists = len(json_files_after) > 0

    if result.returncode == 0 and nifti_exists:
        conversion_status = "converted"
    elif result.returncode == 0 and not nifti_exists:
        conversion_status = "completed_no_nifti_found"
    else:
        conversion_status = "conversion_failed"

    conversion_records.append({
        "RID": row["RID"],
        "subject_id": subject_id,
        "final_group": final_group,
        "image_id": image_id,
        "input_dicom_dir": str(input_dicom_dir),
        "output_nifti_dir": str(output_nifti_dir),
        "output_file_base": output_file_base,
        "conversion_status": conversion_status,
        "return_code": result.returncode,
        "nifti_exists": nifti_exists,
        "json_exists": json_exists,
        "n_nifti_files": len(nifti_files_after),
        "n_json_files": len(json_files_after),
        "nifti_files": "; ".join(str(path) for path in nifti_files_after),
        "json_files": "; ".join(str(path) for path in json_files_after),
        "stdout_tail": result.stdout[-1000:],
        "stderr_tail": result.stderr[-1000:]
    })

    # Save progress regularly so that the report is not lost if the runtime stops.
    if len(conversion_records) % 25 == 0:
        pd.DataFrame(conversion_records).to_csv(
            FULL_CONVERSION_REPORT_PATH,
            index=False
        )
        elapsed_minutes = (time.time() - start_time) / 60
        print(f"Progress saved after {len(conversion_records)} records. Elapsed minutes: {elapsed_minutes:.2f}")

# Save final conversion report
full_conversion_report = pd.DataFrame(conversion_records)
full_conversion_report.to_csv(FULL_CONVERSION_REPORT_PATH, index=False)

elapsed_minutes = (time.time() - start_time) / 60

print("\nSaved full conversion report:")
print(FULL_CONVERSION_REPORT_PATH)

print("\nConversion status counts:")
print(full_conversion_report["conversion_status"].value_counts(dropna=False))

print("\nNIfTI existence:")
print(full_conversion_report["nifti_exists"].value_counts(dropna=False))

print("\nJSON existence:")
print(full_conversion_report["json_exists"].value_counts(dropna=False))

print("\nConverted/skipped files by final group:")
print(
    full_conversion_report
    .groupby(["final_group", "conversion_status"])
    .size()
)

print("\nElapsed time in minutes:")
print(round(elapsed_minutes, 2))

problem_conversions = full_conversion_report[
    ~full_conversion_report["conversion_status"].isin(
        ["converted", "skipped_existing_nifti"]
    )
].copy()

print("\nProblem conversions:")
print(len(problem_conversions))

if len(problem_conversions) > 0:
    display(
        problem_conversions[
            [
                "RID",
                "subject_id",
                "final_group",
                "image_id",
                "conversion_status",
                "return_code",
                "nifti_exists",
                "json_exists",
                "input_dicom_dir",
                "stderr_tail"
            ]
        ]
    )

print("\nAll selected MRI series have NIfTI output:")
print(full_conversion_report["nifti_exists"].all())

## 1.9. Interpretation of Full DICOM-to-NIfTI Conversion

The full DICOM-to-NIfTI conversion was completed successfully for the clean 90-day MRI cohort.

All 1063 selected MRI series now have NIfTI output files, and all 1063 also have accompanying JSON metadata files. No conversion failures were reported.

Some cases were marked as `skipped_existing_nifti`, which means the corresponding NIfTI file already existed before the loop reached that subject. This is expected when pilot conversions or earlier partial conversion runs have already produced output files. These cases are still valid because the final check confirms that NIfTI files exist for every selected MRI series.

The NIfTI files are now ready for post-conversion quality control before further MRI preprocessing.

## 1.10. Post-Conversion NIfTI File QC

After conversion, The notebook checks that every selected MRI has a corresponding NIfTI file and JSON metadata file on disk. I also inspect file sizes to identify empty or unusually small outputs.

This step does not change the data. It verifies that the conversion outputs are physically present and records a QC report for later reference.

In [ ]:
NIFTI_FILE_QC_REPORT_PATH = QC_DIR / "clean_90d_nifti_file_qc_report.csv"

# If the full conversion report is not currently in memory, reload it from disk.
if "full_conversion_report" not in globals():
    full_conversion_report = pd.read_csv(FULL_CONVERSION_REPORT_PATH)

nifti_file_qc = full_conversion_report.copy()

def first_path_from_semicolon_list(path_text):
    if pd.isna(path_text) or str(path_text).strip() == "":
        return None
    return str(path_text).split(";")[0].strip()

nifti_file_qc["primary_nifti_path"] = nifti_file_qc["nifti_files"].apply(first_path_from_semicolon_list)
nifti_file_qc["primary_json_path"] = nifti_file_qc["json_files"].apply(first_path_from_semicolon_list)

nifti_file_qc["primary_nifti_exists"] = nifti_file_qc["primary_nifti_path"].apply(
    lambda x: Path(x).exists() if x is not None else False
)

nifti_file_qc["primary_json_exists"] = nifti_file_qc["primary_json_path"].apply(
    lambda x: Path(x).exists() if x is not None else False
)

nifti_file_qc["nifti_size_mb"] = nifti_file_qc["primary_nifti_path"].apply(
    lambda x: Path(x).stat().st_size / (1024 ** 2) if x is not None and Path(x).exists() else np.nan
)

nifti_file_qc["json_size_kb"] = nifti_file_qc["primary_json_path"].apply(
    lambda x: Path(x).stat().st_size / 1024 if x is not None and Path(x).exists() else np.nan
)

nifti_file_qc["file_qc_status"] = np.where(
    nifti_file_qc["primary_nifti_exists"] & nifti_file_qc["primary_json_exists"] & (nifti_file_qc["nifti_size_mb"] > 1),
    "present_and_nonempty",
    "check_file_output"
)

nifti_file_qc.to_csv(NIFTI_FILE_QC_REPORT_PATH, index=False)

print("Saved NIfTI file QC report:")
print(NIFTI_FILE_QC_REPORT_PATH)

print("\nQC status counts:")
print(nifti_file_qc["file_qc_status"].value_counts(dropna=False))

print("\nNIfTI files present:")
print(nifti_file_qc["primary_nifti_exists"].value_counts(dropna=False))

print("\nJSON files present:")
print(nifti_file_qc["primary_json_exists"].value_counts(dropna=False))

print("\nNIfTI file size summary, MB:")
display(nifti_file_qc["nifti_size_mb"].describe())

print("\nNIfTI counts by final group:")
print(nifti_file_qc["final_group"].value_counts())

problem_nifti_files = nifti_file_qc[
    nifti_file_qc["file_qc_status"] != "present_and_nonempty"
].copy()

print("\nProblem NIfTI file outputs:")
print(len(problem_nifti_files))

if len(problem_nifti_files) > 0:
    display(
        problem_nifti_files[
            [
                "RID",
                "subject_id",
                "final_group",
                "image_id",
                "file_qc_status",
                "primary_nifti_exists",
                "primary_json_exists",
                "nifti_size_mb",
                "primary_nifti_path"
            ]
        ]
    )

## 1.11. Check NIfTI Image Dimensions and Metadata

After confirming that all NIfTI files exist, The notebook checks whether they can be opened successfully and inspect their basic image metadata.

This step reads only the NIfTI headers, not the full image arrays, so it is a lightweight quality-control step. I record each image shape, voxel size, orientation code, and data type. This helps identify unusual outputs such as unreadable files, non-3D images, or unexpected image dimensions before moving to later MRI preprocessing.

In [ ]:
try:
    import nibabel as nib
except ImportError:
    !pip -q install nibabel
    import nibabel as nib

NIFTI_HEADER_QC_REPORT_PATH = QC_DIR / "clean_90d_nifti_header_qc_report.csv"

# Reload file-level QC if needed
if "nifti_file_qc" not in globals():
    nifti_file_qc = pd.read_csv(NIFTI_FILE_QC_REPORT_PATH)

header_qc_records = []

for idx, row in nifti_file_qc.iterrows():
    nifti_path = row["primary_nifti_path"]

    record = {
        "RID": row["RID"],
        "subject_id": row["subject_id"],
        "final_group": row["final_group"],
        "image_id": row["image_id"],
        "nifti_path": nifti_path,
        "load_status": None,
        "shape": None,
        "n_dimensions": None,
        "voxel_sizes": None,
        "orientation": None,
        "data_dtype": None,
        "error_message": ""
    }

    try:
        # nib.load reads the NIfTI header and metadata.
        # It does not load the full image array into memory here.
        img = nib.load(nifti_path)

        shape = img.shape
        voxel_sizes = img.header.get_zooms()[:len(shape)]
        orientation = "".join(nib.aff2axcodes(img.affine))

        record["load_status"] = "loaded"
        record["shape"] = str(shape)
        record["n_dimensions"] = len(shape)
        record["voxel_sizes"] = str(voxel_sizes)
        record["orientation"] = orientation
        record["data_dtype"] = str(img.get_data_dtype())

    except Exception as e:
        record["load_status"] = "load_failed"
        record["error_message"] = str(e)

    header_qc_records.append(record)

nifti_header_qc = pd.DataFrame(header_qc_records)

nifti_header_qc["header_qc_status"] = np.where(
    (nifti_header_qc["load_status"] == "loaded") &
    (nifti_header_qc["n_dimensions"] == 3),
    "loaded_3d_image",
    "check_header_or_dimensions"
)

nifti_header_qc.to_csv(NIFTI_HEADER_QC_REPORT_PATH, index=False)

print("Saved NIfTI header QC report:")
print(NIFTI_HEADER_QC_REPORT_PATH)

print("\nLoad status counts:")
print(nifti_header_qc["load_status"].value_counts(dropna=False))

print("\nHeader QC status counts:")
print(nifti_header_qc["header_qc_status"].value_counts(dropna=False))

print("\nMost common image shapes:")
print(nifti_header_qc["shape"].value_counts().head(20))

print("\nMost common voxel sizes:")
print(nifti_header_qc["voxel_sizes"].value_counts().head(20))

print("\nOrientation counts:")
print(nifti_header_qc["orientation"].value_counts(dropna=False))

print("\nData type counts:")
print(nifti_header_qc["data_dtype"].value_counts(dropna=False))

problem_headers = nifti_header_qc[
    nifti_header_qc["header_qc_status"] != "loaded_3d_image"
].copy()

print("\nProblem NIfTI headers or dimensions:")
print(len(problem_headers))

if len(problem_headers) > 0:
    display(
        problem_headers[
            [
                "RID",
                "subject_id",
                "final_group",
                "image_id",
                "load_status",
                "shape",
                "n_dimensions",
                "error_message",
                "nifti_path"
            ]
        ]
    )

## 1.12. Interpretation of NIfTI Header Quality Control

The NIfTI header quality-control report was saved successfully:

`/content/drive/My Drive/adni_mri/qc/clean_90d_nifti_header_qc_report.csv`

This report stores the basic metadata extracted from each converted NIfTI file, including the subject ID, final clinical group, image ID, file path, image shape, voxel size, orientation, data type, and loading status. This file provides a permanent record of the post-conversion QC step.

### 1.12.1. File Loading Status

All 1063 NIfTI files were successfully loaded:

`loaded = 1063`

This means that every converted `.nii.gz` file could be opened by `nibabel`. Therefore, there is no evidence of corrupted or unreadable NIfTI files at this stage.

### 1.12.2. 3D Image Check

All 1063 images were identified as valid 3D images:

`loaded_3d_image = 1063`

A structural brain MRI should normally be represented as a 3D volume, with dimensions corresponding to width, height, and depth. Therefore, this confirms that all converted images have the expected dimensional structure for MRI analysis.

### 1.12.3. Image Shapes

The image shape describes the number of voxels in each spatial direction. For example, a shape of `(176, 240, 256)` means that the MRI volume contains 176 voxels in one direction, 240 voxels in the second direction, and 256 voxels in the third direction.

The most common shapes were:

* `(176, 240, 256)` for 405 images
* `(160, 192, 192)` for 183 images
* `(208, 240, 256)` for 142 images
* `(170, 256, 256)` for 124 images

The variation in image shape is expected because the ADNI MRI scans come from different acquisition protocols, scanners, ADNI phases, and field strengths. This is not an error at the conversion stage. However, it means that the images are not yet standardized for machine learning. Later preprocessing will need to place the images into a common spatial format.

### 1.12.4. Voxel Sizes

Voxel size describes the physical size of each 3D pixel in millimetres. For example, a voxel size of `(1.0, 1.0, 1.0)` means that each voxel represents a 1 mm × 1 mm × 1 mm cube in physical space.

The most common voxel sizes included:

* `(1.0, 1.0, 1.0)`
* approximately `(1.2, 1.05, 1.05)`
* `(1.2, 1.25, 1.25)`
* approximately `(1.2, 1.0, 1.0)`

The values shown as `np.float32(...)` are just how Python displays floating-point numbers. For example, `np.float32(1.1999999)` can be interpreted as approximately `1.2`.

The variation in voxel size is also expected in raw converted MRI data. For later modelling, the images will need to be resampled or registered into a common voxel resolution or template space.

### 1.12.5. Orientation

Most images are stored in RAS orientation:

`RAS = 1056`

RAS means Right-Anterior-Superior, which is a common orientation convention in neuroimaging.

Seven images have a different orientation:

* `PSR = 6`
* `PIR = 1`

This is not necessarily an error. These images loaded successfully and are valid 3D MRI volumes. However, orientation must be standardized before modelling so that all images are spatially aligned in the same convention. A later preprocessing step should reorient all images to a common orientation, such as RAS.

### 1.12.6. Data Type

The NIfTI files use two intensity storage formats:

* `int16 = 1051`
* `uint16 = 12`

These are both common ways of storing MRI voxel intensity values. This difference is not a reason to exclude images. During preprocessing, image intensities will later be converted into a standard numerical format and normalized before modelling.

### 1.12.7. Problem Headers or Dimensions

No problem headers or dimensions were found:

`Problem NIfTI headers or dimensions = 0`

This means that there were no unreadable files, no non-3D images, and no obvious header-level conversion problems.

### 1.12.8. Overall Interpretation

The NIfTI header QC passed successfully. All 1063 selected MRI scans were converted into readable 3D NIfTI files. Every selected image has a valid NIfTI output, and no conversion or header failures were detected.

The observed differences in image shape, voxel size, orientation, and data type are expected for raw converted MRI scans from ADNI. These differences show that the images are still in their original scanner-space format and need further preprocessing before they can be used for machine learning.

The next step is to move from conversion QC to MRI preprocessing. The immediate next check should document the small number of non-RAS images before applying standardization steps such as reorientation, resampling, registration, cropping or resizing, and intensity normalization.
